# 04 Dodatkowe analizy – walidacja modelu NLI (H4a)

## 1. Konfiguracja

In [ ]:
from google.colab import drive

drive.mount("/content/drive")
%cd /content/drive/MyDrive/Licencjat/extra

import sys
sys.path.append("..")

import pandas as pd
import torch
from sklearn.metrics import cohen_kappa_score, f1_score
from transformers import AutoModelForSequenceClassification, AutoTokenizer

from src.config import LABELS_DIR, NLI_MODEL, PROCESSED_DIR, RESULTS_DIR
from src.nli import nli_probabilities

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BATCH_SIZE = 8
device = "cuda" if torch.cuda.is_available() else "cpu"

## 2. Etykiety i teksty (`labels/stance_dev.csv`, `labels/stance_test.csv`: `id`, `term`, `label`)

In [ ]:
sample = pd.read_parquet(PROCESSED_DIR / "sample.parquet", columns=["id", "text"])
labelled = {}
for name in ("dev", "test"):
    labels = pd.read_csv(LABELS_DIR / f"stance_{name}.csv", dtype={"id": str})
    labelled[name] = labels.merge(sample, on="id")
    print(name, "etykiet:", len(labels), "| znalezionych tekstów:", len(labelled[name]))

## 3. Model

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(NLI_MODEL)
model = AutoModelForSequenceClassification.from_pretrained(NLI_MODEL).to(device).eval()

## 4. Kappa Cohena i F1 dla obu wariantów

In [ ]:
scores = []
for name, data in labelled.items():
    probabilities = nli_probabilities(data["text"].tolist(), data["term"].tolist(), model, tokenizer, device, BATCH_SIZE)
    for variant, column in (("główny: 2 klasy", "critical"), ("porównawczy: 3 klasy", "critical_3")):
        scores.append({"zbiór": name, "wariant": variant, "n": len(data),
                       "kappa": cohen_kappa_score(data["label"], probabilities[column]),
                       "f1": f1_score(data["label"], probabilities[column])})
scores = pd.DataFrame(scores)
scores.to_csv(RESULTS_DIR / "validation_stance.csv", index=False)
scores.round(3)